In [1]:
import re
import pandas as pd
from datetime import datetime


In [2]:
file_path="WhatsApp Chat with Workout.txt"

In [3]:
with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()


In [4]:
clean_lines = []

for line in lines:
    line = line.strip()

    # skip empty lines
    if not line:
        continue

    # remove edited marker but KEEP the message
    line = line.replace("<This message was edited>", "").strip()

    # keep only workout-like messages
    if "set" in line.lower() and "*" in line and "reps" in line.lower():
        clean_lines.append(line)

In [5]:
rows = []

for line in clean_lines:
    try:
        # Example:
        # 13/10/2025, 5:10 pm - Kshitish: Tricep rope overhead set2: 25kg*4reps

        # split datetime and message
        datetime_part, message_part = line.split(" - ", 1)

        # split date and time
        date_str, time_str = datetime_part.split(",", 1)
        date = datetime.strptime(date_str.strip(), "%d/%m/%Y").date()
        time = time_str.strip()

        # remove sender name
        message = message_part.split(":", 1)[1].strip()

        # extract exercise (everything before 'set')
        exercise = message.split("set")[0].strip()

        # extract weight
        weight_match = re.search(r"(\d+\.?\d*)\s*kg", message, re.IGNORECASE)
        if weight_match:
            weight = float(weight_match.group(1))
        else:
            # fallback: number before *
            weight = float(re.search(r"(\d+\.?\d*)\*", message).group(1))

        # extract reps
        reps = int(re.search(r"(\d+)\s*reps", message, re.IGNORECASE).group(1))

        rows.append([date, time, exercise, weight, reps])

    except Exception:
        # if any line breaks parsing, skip safely
        continue

In [6]:
df = pd.DataFrame(
    rows,
    columns=["date", "time", "exercise", "weight", "reps"]
)

In [7]:
df.to_csv("workout_raw_extracted.csv", index=False)



In [8]:
df.head()

,date,time,exercise,weight,reps
0,2025-10-13,5:10 pm,Dumbbell press,20.0,9
1,2025-10-13,5:15 pm,Dumbbell press,20.0,7
2,2025-10-13,5:20 pm,Dumbbell press,20.0,6
3,2025-10-13,5:32 pm,Barbell rowing,20.0,7
4,2025-10-13,5:37 pm,Barbell rowing,20.0,7
